# Plot ROI correlation matrix
Interactive/inline plotting of Shen-268 ROI time-series files.

In [ ]:
# Choose backend: prefer ipympl widgets, fallback to inline
try:
    import importlib
    importlib.import_module("ipympl")
    %matplotlib ipympl
except Exception:
    %matplotlib inline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.covariance import ledoit_wolf


In [ ]:
def cov2corr(covariance: np.ndarray) -> np.ndarray:
    v = np.sqrt(np.diag(covariance))
    outer_v = np.outer(v, v)
    corr = covariance / outer_v
    corr[covariance == 0] = 0
    return corr


def compute_corr(ts: np.ndarray, method: str = "pearson") -> np.ndarray:
    if ts.ndim != 2:
        raise ValueError(f"Expected 2D array (T, N_ROI), got shape {ts.shape}")

    if method == "pearson":
        return np.corrcoef(ts, rowvar=False)
    if method == "ldw":
        cov, _ = ledoit_wolf(ts, assume_centered=False)
        return cov2corr(cov)
    raise ValueError(f"Unknown method: {method}")


def plot_corr(corr: np.ndarray, title: str = "", cmap: str = "coolwarm"):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap=cmap)
    ax.set_title(title or "ROI Correlation Matrix")
    ax.set_xlabel("ROI index")
    ax.set_ylabel("ROI index")
    n_roi = corr.shape[0]
    tick_step = max(1, n_roi // 10)
    ticks = np.arange(0, n_roi, tick_step)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Correlation")
    fig.tight_layout()
    plt.show()


## Load and Process Data
The `input_file_path` parameter is injected by Papermill when run via the backend API.

In [ ]:
# Parameter for Papermill-style execution (local validation)
input_file_path = "../data/100610_MOVIE1_7T_AP_shen268_roi_ts_gsr.txt"


In [ ]:
# Load uploaded file (input_file_path injected by Papermill)
try:
    ts_path = Path(input_file_path)
except NameError:
    # Fallback for manual testing
    ts_path = Path("../data/100610_MOVIE1_7T_AP_shen268_roi_ts_gsr.txt")
    print(f"Warning: Using fallback path for testing: {ts_path}")

if not ts_path.exists():
    raise FileNotFoundError(f"Input file not found: {ts_path}")

method = "pearson"  # or "ldw"

ts = np.loadtxt(ts_path)
if ts.shape[0] == 268 and ts.shape[1] != 268:
    ts = ts.T
print(f"Loaded {ts_path}")
print(f"Time-series shape: {ts.shape} (T x N_ROI)")

corr = compute_corr(ts, method=method)
print("Correlation matrix shape:", corr.shape)
plot_corr(corr, title=f"{ts_path.stem} ({method})")
